In [21]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim. lr_scheduler import CosineAnnealingLR
import numpy as np

In [22]:
# 1. 가상 데이터 생성
def self_study_data():
    X = torch.randn(1000, 10)
    y = torch.randint(0, 2, (1000,))
    dataset = TensorDataset(X, y)

    # Train(80%), Val(10%), Test(10%)
    train_set, val_set, test_set = torch.utils.data.random_split(dataset, [800, 100, 100])
    return DataLoader(train_set, batch_size=32), DataLoader(val_set, batch_size=32), DataLoader(test_set, batch_size=32)

In [23]:
# 2. 간단한 모델 정의
model = nn.Sequential(
nn.Linear (10, 50),
nn.ReLU(),
nn.Linear (50, 21)
)

# 3. 옵티마이저 설정(AdamW)
# weight_decay? L2 규제
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

# 4. 학습률 스케줄러 설정
# T_max? 학습률이 최소가 되는 시점(Epoch)를 의미
scheduler = CosineAnnealingLR(optimizer, T_max=50)

In [50]:
# Early Stopping & Checkpoint
class EarlyStoppingAndCheckpoint:
    def __init__(self, patience=10, min_delta=1e-4, path='best_model.pth'):
        self.patience = patience        # 개선이 없을 때 기다릴 횟수
        self.min_delta = min_delta      # 개선으로 인정할 최소 번화랑
        self.path = path                # 최적 모델 저장 경로
        self.counter = 0                # 기다린 횟수 카운터
        self.best_loss = float('inf')   # 역대 최저 Loss 저장
        self.early_stop = False         # Early Stopping을 사용할지에 대한 플래그

    def __call__(self, val_loss, model, optimizer, scheduler, epoch):
        # 이번 성능이 역대 최고인지 체크
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            # 최고의 epoch를 저장
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'loss': val_loss,
            }, self.path)
        
        # 개선 없으면 카운터 증가, patience 넘으면 early stop
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

In [51]:
# 5. 메인 학습 루프
train_loader, val_loader, _ = self_study_data()
handler = EarlyStoppingAndCheckpoint()
for epoch in range(1, 101):
    # 훈련 단계
    model.train()
    for X_batch, y_batch in train_loader :
        optimizer.zero_grad()
        output = model (X_batch)
        loss = nn.CrossEntropyLoss()(output, y_batch)
        loss.backward()
        optimizer.step()

    # 검증 단계
    model .eval ()
    val_loss = 0
    with torch.no_grad():
        for X_val, y_val in val_loader:
            val_out = model(X_val)
            val_loss += nn.CrossEntropyLoss()(val_out, y_val).item()
    val_loss /= len(val_loader)

    # 스케졸러 & 조기 종료 적용
    scheduler.step()
    current_lr = optimizer.param_groups[0] ['lr']

    print(f"Epoch {epoch} | Val Loss: {val_loss: .4f} | LR: {current_lr : .6f}")

    # 골든 타임 포착 및 조기 종료 체크
    handler(val_loss, model, optimizer, scheduler, epoch)

    if handler.early_stop:
        print("조기 종료: 더 이상 성능 개선이 없어 학습을 종료함")
        break

Epoch 1 | Val Loss:  0.7231 | LR:  0.000281
Epoch 2 | Val Loss:  0.7200 | LR:  0.000277
Epoch 3 | Val Loss:  0.7169 | LR:  0.000271
Epoch 4 | Val Loss:  0.7141 | LR:  0.000266
Epoch 5 | Val Loss:  0.7118 | LR:  0.000259
Epoch 6 | Val Loss:  0.7098 | LR:  0.000253
Epoch 7 | Val Loss:  0.7082 | LR:  0.000246
Epoch 8 | Val Loss:  0.7068 | LR:  0.000238
Epoch 9 | Val Loss:  0.7056 | LR:  0.000230
Epoch 10 | Val Loss:  0.7046 | LR:  0.000222
Epoch 11 | Val Loss:  0.7038 | LR:  0.000214
Epoch 12 | Val Loss:  0.7030 | LR:  0.000205
Epoch 13 | Val Loss:  0.7024 | LR:  0.000196
Epoch 14 | Val Loss:  0.7019 | LR:  0.000187
Epoch 15 | Val Loss:  0.7015 | LR:  0.000178
Epoch 16 | Val Loss:  0.7011 | LR:  0.000169
Epoch 17 | Val Loss:  0.7007 | LR:  0.000159
Epoch 18 | Val Loss:  0.7004 | LR:  0.000150
Epoch 19 | Val Loss:  0.7002 | LR:  0.000141
Epoch 20 | Val Loss:  0.6999 | LR:  0.000131
Epoch 21 | Val Loss:  0.6997 | LR:  0.000122
Epoch 22 | Val Loss:  0.6996 | LR:  0.000113
Epoch 23 | Val Loss